# Partie II — CNN et vision par ordinateur (CIFAR-10)

Classification d'images **couleur** avec un réseau convolutif (CNN) de type LeNet.
On utilise **CIFAR-10** : 60 000 images 32×32 en RGB, réparties en 10 classes
(avion, voiture, oiseau, chat, cerf, chien, grenouille, cheval, bateau, camion).

Par rapport à MNIST, ce dataset est plus difficile : images **couleur** (3 canaux),
objets naturels variés, fond non uniforme. Cela oblige à adapter l'architecture
(3 canaux d'entrée) et rend la tâche plus représentative d'un vrai problème de vision.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

torch.manual_seed(42)  # reproductibilité
np.random.seed(42)

print("PyTorch :", torch.__version__)
print("GPU disponible :", torch.cuda.is_available())

## 1. Chargement et préparation des données

On normalise chaque canal RGB (moyenne 0.5, écart-type 0.5) pour stabiliser
l'apprentissage. Les données sont déjà séparées en apprentissage / test par CIFAR-10.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # 3 canaux RGB
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                             download=True, transform=transform)
test_dataset  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                             download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

classes = ['avion', 'voiture', 'oiseau', 'chat', 'cerf',
           'chien', 'grenouille', 'cheval', 'bateau', 'camion']

print(f"Train : {len(train_dataset)} images | Test : {len(test_dataset)} images")
print(f"Forme d'une image : {train_dataset[0][0].shape}  (canaux, hauteur, largeur)")

In [ ]:
# Visualisation de quelques exemples
images, labels = next(iter(train_loader))

def imshow(img):
    img = img / 2 + 0.5            # dé-normalisation pour l'affichage
    return np.transpose(img.numpy(), (1, 2, 0))

plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(imshow(images[i]))
    plt.title(f"{classes[labels[i].item()]}")
    plt.axis('off')
plt.suptitle("Exemples d'images CIFAR-10")
plt.tight_layout()
plt.show()

print(f"Forme d'un batch : {images.shape}")

## 2. Calculs dimensionnels

Taille de sortie d'une convolution : `out = (in + 2·padding − kernel) // stride + 1`.
Pour le pooling, le stride vaut par défaut la taille du noyau.

In [ ]:
def conv_output_size(input_size, kernel_size, padding=0, stride=1):
    return (input_size + 2*padding - kernel_size) // stride + 1

def pool_output_size(input_size, kernel_size, stride=None):
    if stride is None:
        stride = kernel_size
    return (input_size - kernel_size) // stride + 1

print("=== Calculs dimensionnels pour CIFAR-10 (32x32) ===\n")
s1 = conv_output_size(32, kernel_size=5, padding=2)   # conv1
p1 = pool_output_size(s1, 2)                           # pool1
s2 = conv_output_size(p1, kernel_size=5, padding=0)    # conv2
p2 = pool_output_size(s2, 2)                           # pool2
print(f"Entrée 32x32")
print(f"  conv1 (k=5, p=2) -> {s1}x{s1}")
print(f"  pool1 (2x2)      -> {p1}x{p1}")
print(f"  conv2 (k=5, p=0) -> {s2}x{s2}")
print(f"  pool2 (2x2)      -> {p2}x{p2}")
print(f"  -> vecteur aplati : 16 x {p2} x {p2} = {16*p2*p2}")

## 3. Implémentation manuelle des opérations convolutionnelles

On programme « à la main » la corrélation croisée 2D et le max-pooling, puis on
vérifie qu'elles donnent le même résultat que PyTorch. Rappel : ce que les CNN
appellent convolution est en réalité une **corrélation croisée** (pas de retournement du noyau).

In [ ]:
def correlation_croisee_2d(input_map, kernel):
    """Corrélation croisée 2D implémentée manuellement."""
    h_in, w_in = input_map.shape
    h_k, w_k = kernel.shape
    h_out, w_out = h_in - h_k + 1, w_in - w_k + 1
    output = np.zeros((h_out, w_out))
    for i in range(h_out):
        for j in range(w_out):
            output[i, j] = np.sum(input_map[i:i+h_k, j:j+w_k] * kernel)
    return output

# Vérification face à PyTorch
X = np.array([[0,1,2],[3,4,5],[6,7,8]], dtype=np.float32)
K = np.array([[0,1],[2,3]], dtype=np.float32)
manuel = correlation_croisee_2d(X, K)

conv = nn.Conv2d(1, 1, kernel_size=2, bias=False)
conv.weight.data = torch.tensor(K).reshape(1,1,2,2)
pytorch = conv(torch.tensor(X).reshape(1,1,3,3)).detach().numpy().reshape(2,2)

print("Manuel :\n", manuel)
print("PyTorch :\n", pytorch)
print("Identiques :", np.allclose(manuel, pytorch))

In [ ]:
def max_pooling_2d(input_map, kernel_size=2, stride=2):
    """Max-pooling implémenté manuellement."""
    h_in, w_in = input_map.shape
    h_out = (h_in - kernel_size) // stride + 1
    w_out = (w_in - kernel_size) // stride + 1
    output = np.zeros((h_out, w_out))
    for i in range(h_out):
        for j in range(w_out):
            output[i, j] = np.max(input_map[i*stride:i*stride+kernel_size,
                                            j*stride:j*stride+kernel_size])
    return output

Xp = np.arange(16, dtype=np.float32).reshape(4, 4)
manuel_pool = max_pooling_2d(Xp, 2, 2)
pytorch_pool = nn.MaxPool2d(2, 2)(torch.tensor(Xp).reshape(1,1,4,4)).numpy().reshape(2,2)
print("Max-pooling manuel :\n", manuel_pool)
print("Identiques à PyTorch :", np.allclose(manuel_pool, pytorch_pool))

## 4. Modèle CNN (LeNet adapté à CIFAR-10)

L'architecture LeNet est adaptée aux images couleur : la première convolution
prend **3 canaux** en entrée (RGB) au lieu d'un seul. On utilise le max-pooling
(qui conserve les activations saillantes).

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        # Bloc convolutionnel 1 : 3 canaux d'entrée (RGB)
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5, padding=2)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Bloc convolutionnel 2
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Classifieur entièrement connecté
        self.fc1 = nn.Linear(16*6*6, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)        # aplatir
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

lenet = LeNet().to(device)
print(lenet)
print("Nombre de paramètres :", sum(p.numel() for p in lenet.parameters()))

## 5. Entraînement

Optimiseur Adam, perte CrossEntropy. On suit la perte d'apprentissage et
l'accuracy de test à chaque époque.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lenet.parameters(), lr=0.001)

n_epochs = 8
train_losses, test_accuracies = [], []

for epoch in range(n_epochs):
    lenet.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = lenet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_losses.append(running_loss / len(train_loader))

    # évaluation sur le test
    lenet.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            preds = lenet(images).argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    test_accuracies.append(correct / total)
    print(f"Époque {epoch+1}/{n_epochs} | loss {train_losses[-1]:.4f} | "
          f"accuracy test {test_accuracies[-1]:.4f}")

In [ ]:
# Courbes d'apprentissage
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, color='steelblue')
plt.title("Loss d'entraînement"); plt.xlabel("Époque"); plt.ylabel("Loss")
plt.subplot(1, 2, 2)
plt.plot(test_accuracies, color='green')
plt.title("Accuracy sur le test"); plt.xlabel("Époque"); plt.ylabel("Accuracy")
plt.tight_layout(); plt.show()

## 6. Évaluation détaillée

Matrice de confusion et rapport de classification (précision, rappel, F1 par classe).

In [ ]:
lenet.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = lenet(images).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title("Matrice de confusion — CNN (CIFAR-10)")
plt.ylabel("Vraie classe"); plt.xlabel("Classe prédite")
plt.tight_layout(); plt.show()

print(classification_report(all_labels, all_preds, target_names=classes))

## 7. Visualisation des cartes de caractéristiques

On extrait les sorties de la première couche convolutionnelle via un *hook*,
pour observer ce que les filtres détectent (bords, couleurs, textures).

In [ ]:
img, label = test_dataset[0]
img_tensor = img.unsqueeze(0).to(device)

activation = {}
def get_activation(name):
    def hook(model, inp, output):
        activation[name] = output.detach()
    return hook

lenet.conv1.register_forward_hook(get_activation('conv1'))
_ = lenet(img_tensor)
feature_maps = activation['conv1'].squeeze(0).cpu()

plt.figure(figsize=(12, 3))
plt.subplot(1, 7, 1)
plt.imshow(imshow(img)); plt.title(f"{classes[label]}"); plt.axis('off')
for k in range(6):
    plt.subplot(1, 7, k+2)
    plt.imshow(feature_maps[k], cmap='viridis')
    plt.title(f"Filtre {k}"); plt.axis('off')
plt.suptitle("Cartes de caractéristiques — 1re couche conv")
plt.tight_layout(); plt.show()

## 8. Comparaison MLP vs CNN

Un MLP simple (image aplatie en vecteur de 3×32×32 = 3072) sert de point de
comparaison. On s'attend à ce que le CNN le dépasse, car il exploite la
structure spatiale que le MLP ignore.

In [ ]:
class MLP_Images(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3*32*32, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = x.view(x.size(0), -1)   # aplatir l'image couleur
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

mlp = MLP_Images().to(device)
optim_mlp = torch.optim.Adam(mlp.parameters(), lr=0.001)

for epoch in range(8):
    mlp.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optim_mlp.zero_grad()
        loss = criterion(mlp(images), labels)
        loss.backward(); optim_mlp.step()

mlp.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = mlp(images).argmax(1)
        correct += (preds == labels).sum().item(); total += labels.size(0)
mlp_acc = correct / total

print(f"Accuracy MLP : {mlp_acc:.4f}  ({sum(p.numel() for p in mlp.parameters())} paramètres)")
print(f"Accuracy CNN : {test_accuracies[-1]:.4f}  ({sum(p.numel() for p in lenet.parameters())} paramètres)")
print("-> Le CNN exploite la structure spatiale (localité, partage des poids),")
print("   ce qui le rend plus performant et plus parcimonieux que le MLP.")

## Conclusion

Sur CIFAR-10, plus difficile que MNIST, le CNN dépasse le MLP en exploitant la
structure spatiale des images couleur. Les cartes de caractéristiques montrent
que les filtres apprennent des détecteurs locaux (contours, couleurs). Les choix
architecturaux (padding pour préserver les bords, max-pooling pour la saillance,
nombre de filtres pour la capacité) influencent directement la performance.

*Pistes d'amélioration : architecture plus profonde, data augmentation, batch
normalization — qui font gagner plusieurs points sur CIFAR-10.*